# Medallion Architecture — Bronze → Silver → Gold

Building a production-grade data pipeline together: from raw CSV/JSON files to business-ready analytics tables.

**This demo uses real notebooks from `materials/medallion/` — 6 production-ready pipeline notebooks we will run and examine together.**

## Learning Objectives

After completing this module you will be able to:

- **Describe** the purpose and design principles of each Medallion layer (Bronze, Silver, Gold)
- **Explain** why raw data cannot be used directly in analytics
- **Apply** Bronze ingestion patterns: append-only, metadata columns, no transformation
- **Implement** Silver transformations: deduplication, type casting, data quality, derived columns
- **Design** Gold aggregation tables: joins, groupBy, KPI computation
- **Trace** data lineage from raw file to analytical result

## Business Scenario: RetailHub E-Commerce

**Context:** RetailHub operates an e-commerce platform. A new data engineering team has been asked to build the analytics foundation.

**Available raw data:**
| Source | Format | Location | Content |
|--------|--------|----------|---------|
| CRM system | CSV | `dataset/customers/` | Customer records (name, email, city, segment) |
| Order system | JSON | `dataset/orders/` | Order transactions (items, quantities, prices, discounts) |

**Business requirements:**
- Analysts need a **360-degree customer view** with total spend, order count, average order value
- Finance needs **daily revenue reports** with trend analysis
- Data quality must be enforced — no NULLs in key fields, no negative quantities

**The problem with raw data:**
- Duplicates from system exports
- Inconsistent email formatting (`ANNA@EXAMPLE.COM`, `anna@example.com`)
- Missing values in critical columns
- No derived business metrics (total_amount, discount_amount)
- No data lineage (which file, when loaded)

**Solution:** Medallion Architecture — three layers of increasing refinement.

## Architecture Overview

```
RAW FILES                BRONZE                   SILVER                    GOLD
─────────────           ──────────────────        ────────────────────      ─────────────────────────────
customers.csv   ──────► bronze_customers   ──────► silver_customers   ──┬──► gold_customer_orders_summary
orders_batch.json ────► bronze_orders      ──────► silver_orders_clean ─┤
                                                                         └──► gold_daily_orders
```

| Layer | Purpose | Write Mode | Quality | Consumers |
|-------|---------|-----------|---------|-----------|
| **Bronze** | Land raw data as-is | Append only | None | Silver jobs |
| **Silver** | Cleanse and conform | Merge / Overwrite | Validated | Gold jobs, data scientists |
| **Gold** | Business-ready aggregations | Overwrite | Trusted | BI tools, dashboards, ML |

**Key principle:** Each layer adds value. Bronze = landing zone. Silver = cleansed truth. Gold = business answers.

![Medallion Architecture — Unified Data Staging Process](../../../assets/images/a433428da3144cfea92e0681ab5c73a9.png)


## Setup

In [0]:
%run ../../setup/00_setup

In [0]:
# Derive path to medallion notebooks dynamically
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
training_root = notebook_path.rsplit('/notebooks/', 1)[0]
MEDALLION_PATH = f"{training_root}/materials/medallion"

print(f"Training root:     {training_root}")
print(f"Medallion path:    {MEDALLION_PATH}")
print(f"Catalog:           {CATALOG}")
print(f"Dataset path:      {DATASET_PATH}")

In [0]:
# Ensure Bronze / Silver / Gold schemas exist
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    print(f"[OK] Schema: {CATALOG}.{schema}")

## Layer 1: Bronze — Land Raw Data

> **Rule:** Store everything exactly as received. No transformations. Add metadata only.

### Why Bronze?

Raw operational systems export data that is:
- **Unvalidated** — no quality checks at source
- **Unstable** — schema can change without notice
- **Unrepeatable** — you may not be able to re-fetch historical data

Bronze is the **safety net** — if Silver processing fails, we can replay from Bronze.

### The Raw Sources

Before building Bronze, let's look at what the source files actually contain.

In [0]:
# Preview raw CSV — customers (first 5 rows)
raw_customers = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{DATASET_PATH}/customers/customers.csv")
)
print(f"Raw customers: {raw_customers.count()} rows, {len(raw_customers.columns)} columns")
display(raw_customers.limit(5))

In [0]:
# Preview raw JSON — orders (first 5 rows)
raw_orders = (
    spark.read
    .format("json")
    .option("inferSchema", "true")
    .load(f"{DATASET_PATH}/orders/orders_batch.json")
)
print(f"Raw orders: {raw_orders.count()} rows, {len(raw_orders.columns)} columns")
display(raw_orders.limit(5))

### Discussion: What Do We See?

**Questions to explore with participants:**
1. Is there a `_load_ts` timestamp? → No, we don't know *when* data was loaded
2. Is there a `_source_file` column? → No lineage to the source file
3. Are there any nulls? → `raw_customers.filter("customer_id IS NULL").count()`
4. Are there duplicates? → Can we tell just from Bronze? (No — that's Silver's job)

> **Instructor tip:** Run `raw_customers.printSchema()` to show inferred types. Ask participants: "Would you trust `inferSchema=True` in production?" (Answer: No — it can change.)

**What Bronze adds:**
- `_load_ts` — timestamp of ingestion
- `_source_file` — lineage back to the file (via `input_file_name()`)
- Nothing else — the rest is Silver's responsibility

### Bronze: Customers — Notebook Walkthrough

**Notebook:** `materials/medallion/bronze_customers.ipynb`

```python
# Parameters (passed from Job or dbutils.notebook.run)
catalog = dbutils.widgets.get("catalog")       # e.g. "retailhub_trainer"
schema  = dbutils.widgets.get("schema")        # "bronze"
source_path = dbutils.widgets.get("source_path")  # path to dataset Volume

target_table = f"{catalog}.{schema}.bronze_customers"

# Read CSV → add metadata → write to Delta (overwrite)
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{source_path}/customers/customers.csv")
    .withColumn("_load_ts", current_timestamp())
)
df.write.format("delta").mode("overwrite").saveAsTable(target_table)
```

**Design decisions:**
- `mode("overwrite")` — customers are a dimension, full refresh is fine
- No transformation — `inferSchema` is acceptable here because Silver will re-cast
- `_load_ts` is the only addition — minimal footprint

In [0]:
# Run: Bronze Customers
result = dbutils.notebook.run(
    f"{MEDALLION_PATH}/bronze_customers",
    timeout_seconds=300,
    arguments={
        "catalog": CATALOG,
        "schema": BRONZE_SCHEMA,
        "source_path": DATASET_PATH
    }
)
import json
print(json.loads(result))

In [0]:
# Inspect Bronze Customers result
bronze_customers_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_customers")
print(f"bronze.bronze_customers: {bronze_customers_df.count()} rows")
display(bronze_customers_df.limit(8))

### Bronze: Orders — Notebook Walkthrough

**Notebook:** `materials/medallion/bronze_orders.ipynb`

```python
# Read JSON → add _load_ts → append to Delta
df = (
    spark.read.format("json")
    .option("inferSchema", "true")
    .load(f"{source_path}/orders/orders_batch.json")
    .withColumn("_load_ts", current_timestamp())
)
df.write.format("delta").mode("append").saveAsTable(target_table)
```

**Key difference from Customers:**
- `mode("append")` — orders are facts, never overwrite (new batches accumulate)
- JSON format handles nested structures naturally

> **Instructor tip:** Ask participants: "Why `append` for orders but `overwrite` for customers?" → Because orders are historical facts (immutable), customers are a current-state dimension.

In [0]:
# Run: Bronze Orders
result = dbutils.notebook.run(
    f"{MEDALLION_PATH}/bronze_orders",
    timeout_seconds=300,
    arguments={
        "catalog": CATALOG,
        "schema": BRONZE_SCHEMA,
        "source_path": DATASET_PATH
    }
)
print(json.loads(result))

In [0]:
# Inspect Bronze Orders result
bronze_orders_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_orders")
print(f"bronze.bronze_orders: {bronze_orders_df.count()} rows")
display(bronze_orders_df.limit(8))

### Bronze Layer — Design Decisions Summary

| Decision | Choice | Reason |
|----------|--------|--------|
| **Schema enforcement** | `inferSchema=True` | Silver will cast to correct types |
| **Customers write mode** | `overwrite` | Dimension — full refresh from CRM |
| **Orders write mode** | `append` | Facts — historical, never delete |
| **Added columns** | `_load_ts` only | Lineage without transformation |
| **Validation** | None | Capture everything — quality is Silver's job |
| **Format** | Delta | Time travel, schema evolution, ACID |

## Layer 2: Silver — Cleanse & Conform

> **Rule:** Apply business rules. Validate quality. Conform schema. One source of truth.

Silver is where data becomes **trustworthy**. Analytics teams read from Silver, not Bronze.

### Discussion: What Problems Exist in Bronze?

Before revealing the Silver notebooks, explore the bronze data with participants.

**Questions to investigate:**

1. Are there NULL `customer_id` values in orders?
2. Are emails consistently formatted?
3. Are there invalid quantities (≤ 0)?
4. Is `total_amount` computed anywhere?

> **Instructor tip:** Explore live — let participants suggest SQL queries to find these issues.

In [0]:
# Explore bronze data quality issues together
from pyspark.sql import functions as F

print("=== Bronze Customers: Data Quality Check ===")
bronze_cust = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_customers")

issues = bronze_cust.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer_ids"),
    F.countDistinct("customer_id").alias("distinct_customers"),
    F.sum(F.when(F.col("email") != F.lower(F.trim(F.col("email"))), 1).otherwise(0)).alias("non_lowercase_emails")
)
display(issues)

In [0]:
print("=== Bronze Orders: Data Quality Check ===")
bronze_ord = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_orders")

issues_ord = bronze_ord.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.col("order_id").isNull().cast("int")).alias("null_order_ids"),
    F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer_ids"),
    F.sum((F.col("quantity") <= 0).cast("int")).alias("invalid_quantities"),
    F.sum(F.col("unit_price").isNull().cast("int")).alias("null_prices")
)
display(issues_ord)
print()
print("Bronze orders schema:")
bronze_ord.printSchema()

### Silver: Customers — Deduplication + Standardization

**Notebook:** `materials/medallion/silver_customers.ipynb`

**Transformations applied:**

```python
from pyspark.sql.window import Window

w = Window.partitionBy("customer_id").orderBy(col("_load_ts").desc())

df_clean = (
    df
    .filter(col("customer_id").isNotNull())       # quality gate: drop nulls
    .withColumn("_rn", row_number().over(w))       # deduplicate: rank by load time
    .filter(col("_rn") == 1)                       # keep latest version only
    .drop("_rn")
    .withColumn("email", lower(trim(col("email")))) # standardize: lowercase + trim
    .withColumn("_processed_at", current_timestamp())
)
df_clean.write.format("delta").mode("overwrite").saveAsTable(target_table)
```

**Key transformations:**
- `filter(customer_id.isNotNull())` — enforce NOT NULL on business key
- `row_number().over(Window.partitionBy("customer_id"))` — deduplication by recency
- `lower(trim(email))` — standardize email format
- `_processed_at` — Silver lineage timestamp

In [0]:
# Run: Silver Customers
result = dbutils.notebook.run(
    f"{MEDALLION_PATH}/silver_customers",
    timeout_seconds=300,
    arguments={
        "catalog": CATALOG,
        "schema_bronze": BRONZE_SCHEMA,
        "schema_silver": SILVER_SCHEMA
    }
)
print(json.loads(result))

In [0]:
# Compare: Bronze vs Silver Customers
bronze_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_customers").count()
silver_cust = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_customers")
silver_count = silver_cust.count()
print(f"Bronze: {bronze_count} rows → Silver: {silver_count} rows (deduplication removed {bronze_count - silver_count})")
display(silver_cust.limit(8))

### Silver: Orders Cleaned — Quality Checks + Derived Columns

**Notebook:** `materials/medallion/silver_orders_cleaned.ipynb`

**Transformations applied:**

```python
df_cleaned = (
    df
    .filter(col("order_id").isNotNull())           # required business key
    .filter(col("customer_id").isNotNull())         # referential integrity
    .filter(col("quantity") > 0)                   # business rule: positive quantities
    .withColumn("total_amount",
        round(col("quantity") * col("unit_price"), 2))
    .withColumn("discount_amount",
        round(col("quantity") * col("unit_price") * col("discount_percent") / 100, 2))
    .withColumn("net_amount",
        col("total_amount") - col("discount_amount"))
    .withColumn("_processed_at", current_timestamp())
)
df_cleaned.write.format("delta").mode("overwrite").saveAsTable(target_table)
```

**Key transformations:**
- Three filter rules = data quality contract
- `total_amount`, `discount_amount`, `net_amount` — business metrics computed once, reused everywhere
- No joins at Silver — keep it simple and fast

> **Instructor tip:** Ask: "Why compute `net_amount` here instead of Gold?" → Because multiple Gold tables need it. Compute once in Silver, use many times.

In [0]:
# Run: Silver Orders Cleaned
result = dbutils.notebook.run(
    f"{MEDALLION_PATH}/silver_orders_cleaned",
    timeout_seconds=300,
    arguments={
        "catalog": CATALOG,
        "schema_bronze": BRONZE_SCHEMA,
        "schema_silver": SILVER_SCHEMA
    }
)
print(json.loads(result))

In [0]:
# Inspect Silver Orders — verify quality and derived columns
silver_orders = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_orders_cleaned")
print(f"silver.silver_orders_cleaned: {silver_orders.count()} rows")
display(silver_orders.select("order_id","customer_id","quantity","unit_price","discount_percent","total_amount","discount_amount","net_amount").limit(8))

### Silver Layer — Design Decisions Summary

| Decision | Choice | Reason |
|----------|--------|--------|
| **Deduplication** | Window function + `row_number()` | Deterministic, handles late arrivals |
| **Null handling** | `filter(col.isNotNull())` | Hard reject on business keys |
| **Email normalization** | `lower(trim(col))` | Single standard for joins and lookups |
| **Derived columns** | Compute in Silver | Reused across multiple Gold tables |
| **Write mode** | `overwrite` | Silver = current state; full refresh from Bronze |
| **No joins** | Pure single-source transforms | Joins belong in Gold or Lakeflow |

### Slowly Changing Dimensions (SCD) — Customer Dimension in Context

When a customer changes their address or email, how should Silver store that change?

| Strategy | Name | Behavior | History | Our pipeline |
|---|---|---|---|---|
| **Type 1** | Overwrite | New value replaces old | Lost | `silver_customers` (current) |
| **Type 2** | Add row | New row added; old row flagged `is_current = False` | Preserved | Would require MERGE + extra columns |

**Type 1** (what `silver_customers.ipynb` implements):
- `row_number().over(Window.partitionBy("customer_id").orderBy(_load_ts desc))` → keep latest
- Simple, fast, no schema overhead
- Use when: only current state matters (profile data, preferences)

**Type 2** (when you need full history):
- Use when: customer address must match the address _at the time of order_ (compliance, billing)
- Requires: `is_current BOOLEAN`, `start_date DATE`, `end_date DATE`
- Implemented via `MERGE INTO`:

```sql
MERGE INTO silver.silver_customers AS target
USING bronze.bronze_customers      AS source
ON target.customer_id = source.customer_id
   AND target.is_current = true
WHEN MATCHED AND (target.email != source.email OR target.address != source.address) THEN
    UPDATE SET is_current = false, end_date = current_date()
WHEN NOT MATCHED THEN
    INSERT (customer_id, first_name, last_name, email, address,
            is_current, start_date, end_date)
    VALUES (source.customer_id, source.first_name, source.last_name,
            source.email, source.address, true, current_date(), null)
```

> **Exam tip:** DEA exam tests whether you can identify SCD Type 1 vs Type 2 from a description. Key signal: *"full history of changes"* → Type 2.


### AUTO CDC — `APPLY CHANGES INTO` (Lakeflow Pipelines Preview)

> **Definition (Databricks):** `APPLY CHANGES INTO` is the Lakeflow Pipelines (DLT) API for CDC (Change Data Capture). It automatically handles SCD Type 1 and Type 2 without writing `MERGE` manually. Available only inside a Lakeflow pipeline (`.py` file with `import dlt`).

```python
import dlt
from pyspark.sql.functions import col

# Step 1: Define the Bronze streaming table (source)
@dlt.table(name="bronze_customers")
def bronze_customers():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .load("/Volumes/retail/landing/customers/")
    )

# Step 2: Apply CDC — SCD Type 1 (overwrite on change)
dlt.apply_changes(
    target      = "silver_customers",       # destination table
    source      = "bronze_customers",       # streaming source
    keys        = ["customer_id"],          # business key
    sequence_by = col("_load_ts"),          # latest version wins (ordering column)
    stored_as_scd_type = 1                  # 1 = overwrite | 2 = add row with is_current flag
)
```

**What `APPLY CHANGES INTO` replaces vs. manual approach:**

| Manual (`silver_customers.ipynb`) | AUTO CDC (`dlt.apply_changes`) |
|---|---|
| `row_number().over(Window...)` | `sequence_by = col("_load_ts")` |
| `filter(_rn == 1)` | handled automatically |
| `mode("overwrite")` | `stored_as_scd_type = 1` |
| Need to write Type 2 MERGE manually | `stored_as_scd_type = 2` |

> **Instructor note:** This is a preview — full hands-on `dlt.apply_changes` is covered in **Day 3: Module 07 — Lakeflow Pipelines**.


## Layer 3: Gold — Business-Ready Analytics

> **Rule:** Answer specific business questions. Optimized for consumption. No raw data.

Gold tables are designed for **specific audiences**: BI dashboards, ML features, finance reports.
Each Gold table answers one question well.

### Business Questions → Gold Tables

| Business Question | Gold Table | Source |
|---|---|---|
| "What's the total spend per customer? Who are our best customers?" | `gold_customer_orders_summary` | silver_customers + silver_orders_cleaned |
| "How does daily revenue trend? Any seasonality?" | `gold_daily_orders` | silver_orders_cleaned |

> **Design principle:** One Gold table = one business question. Never build a "universal" Gold table that tries to answer everything — it ends up serving no one well.

### Gold: Customer Orders Summary — Join + Aggregate

**Notebook:** `materials/medallion/gold_customer_orders_summary.ipynb`

```python
gold_df = (
    orders
    .join(customers, "customer_id", "left")       # enrich with customer attributes
    .groupBy("customer_id", "first_name", "last_name", "customer_segment")
    .agg(
        count("order_id").alias("total_orders"),
        sum("net_amount").alias("total_revenue"),
        avg("net_amount").alias("avg_order_value"),
        max("order_date").alias("last_order_date")
    )
)
gold_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
```

**This Gold table enables:**
- Customer segmentation analysis
- Lifetime value ranking (`ORDER BY total_revenue DESC`)
- Churn detection (`WHERE last_order_date < 90 days ago`)

In [0]:
# Run: Gold Customer Orders Summary
result = dbutils.notebook.run(
    f"{MEDALLION_PATH}/gold_customer_orders_summary",
    timeout_seconds=300,
    arguments={
        "catalog": CATALOG,
        "schema_silver": SILVER_SCHEMA,
        "schema_gold": GOLD_SCHEMA
    }
)
print(json.loads(result))

In [0]:
# Top customers by revenue
gold_summary = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_customer_orders_summary")
print(f"gold.gold_customer_orders_summary: {gold_summary.count()} rows")
display(gold_summary.orderBy("total_revenue", ascending=False).limit(10))

### Gold: Daily Orders — Revenue Trend

**Notebook:** `materials/medallion/gold_daily_orders.ipynb`

```python
gold_df = (
    df
    .withColumn("order_date", to_date(col("order_datetime")))
    .groupBy("order_date")
    .agg(
        count("order_id").alias("order_count"),
        sum("net_amount").alias("total_revenue"),
        round(avg("net_amount"), 2).alias("avg_order_value")
    )
    .orderBy("order_date")
)
gold_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
```

**This Gold table enables:**
- Daily/weekly/monthly revenue dashboards
- Anomaly detection (revenue drop alerts)
- Capacity planning (order volume trends)

In [0]:
# Run: Gold Daily Orders
result = dbutils.notebook.run(
    f"{MEDALLION_PATH}/gold_daily_orders",
    timeout_seconds=300,
    arguments={
        "catalog": CATALOG,
        "schema_silver": SILVER_SCHEMA,
        "schema_gold": GOLD_SCHEMA
    }
)
print(json.loads(result))

In [0]:
# Daily revenue trend
gold_daily = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_orders")
print(f"gold.gold_daily_orders: {gold_daily.count()} rows")
display(gold_daily.orderBy("order_date"))

### Gold Layer — Design Decisions Summary

| Decision | Choice | Reason |
|----------|--------|--------|
| **Joins** | `left join` customers onto orders | Keep all orders, enrich with customer attributes |
| **Granularity** | `groupBy(customer_id)` / `groupBy(order_date)` | One row per business entity |
| **Write mode** | `overwrite` | Gold = current snapshot, always rebuilt from Silver |
| **No raw columns** | Only business metrics | Consumers should never see `_load_ts` or raw types |
| **Naming** | `total_revenue`, `avg_order_value` | Business language, not technical |

## Summary: Medallion Architecture Decision Matrix

| Aspect | Bronze | Silver | Gold |
|--------|--------|--------|------|
| **Purpose** | Landing zone | Cleansed truth | Business answers |
| **Source** | Raw files (CSV, JSON) | Bronze tables | Silver tables |
| **Write mode** | Append (facts) / Overwrite (dims) | Overwrite | Overwrite |
| **Schema** | Inferred / raw types | Strong types, standardized | Denormalized, business names |
| **Quality rules** | None | Hard rejects (NOT NULL, range) | Assumed clean (from Silver) |
| **Transformations** | Add `_load_ts` only | Dedup, cast, normalize, derive | Join, aggregate, compute KPIs |
| **Consumers** | Silver jobs | Gold jobs, data scientists | BI tools, dashboards, ML |
| **Row count** | = source rows | ≤ Bronze (dedup removes rows) | < Silver (aggregated) |
| **History** | Full — every load | Current state | Current state |

**Data flow recap:**
```
customers.csv  ──► bronze_customers (500 rows)  ──► silver_customers (490 rows, deduped)  ──► gold_customer_orders_summary (490 rows, enriched)
orders.json    ──► bronze_orders    (850 rows)  ──► silver_orders_cleaned (840 rows, QA)  ──► gold_daily_orders (30 rows, by day)
```

![Medallion Architecture — Data Processing Pipeline & Star Schema](../../../assets/images/555ecf8e271a4b4381567ef3e1b5a415.png)


## Notebook Modularity Pattern

This demo runs **6 independent notebooks** from a single orchestrator via `dbutils.notebook.run()`.
Each notebook follows the same contract: one input → one output table.

```
06_medallion_architecture.ipynb  (Orchestrator)
│
│  dbutils.notebook.run("bronze_customers",  arguments={...})
│  dbutils.notebook.run("bronze_orders",     arguments={...})
│  dbutils.notebook.run("silver_customers",  arguments={...})
│  dbutils.notebook.run("silver_orders_...", arguments={...})
│  dbutils.notebook.run("gold_customer_...", arguments={...})
│  dbutils.notebook.run("gold_daily_orders", arguments={...})
│
└──► materials/medallion/
        bronze_customers.ipynb          → bronze.bronze_customers
        bronze_orders.ipynb             → bronze.bronze_orders
        silver_customers.ipynb          → silver.silver_customers
        silver_orders_cleaned.ipynb     → silver.silver_orders_cleaned
        gold_customer_orders_summary.ipynb → gold.gold_customer_orders_summary
        gold_daily_orders.ipynb         → gold.gold_daily_orders
```

**Rules that make this pattern work:**

| Rule | Implementation |
|---|---|
| One input → one output | Each notebook reads one table, writes one table |
| Arguments via widgets | `dbutils.widgets.get("catalog")` — testable in isolation |
| Structured return | `dbutils.notebook.exit(json.dumps({"status": "ok", "rows": n}))` |
| No hardcoded paths | Catalog, schema, source path all passed as arguments |
| Idempotent | Run multiple times = same result (`overwrite` / `append` by design) |

**Testing a single notebook in isolation:**
```python
# Run just the silver_customers step with test catalog
dbutils.notebook.run(
    "materials/medallion/silver_customers",
    timeout_seconds=120,
    arguments={
        "catalog": "dev_catalog",        # override to test environment
        "schema_bronze": "bronze",
        "schema_silver": "silver"
    }
)
```

> **Day 3 connection:** In Lakeflow Pipelines, `dbutils.notebook.run()` is replaced by `@dlt.table` decorators. The same modularity principle applies — one decorated function = one managed streaming table. Databricks handles the dependency graph automatically.


## What's Next — Day 3

We just ran 6 notebooks **manually** in sequence.

**Problems with the manual approach:**
- What if `bronze_customers` fails? `silver_customers` would run on stale data
- No retry logic, no alerting, no monitoring
- Runs once — what about daily/hourly refresh?

**Day 3 will address this:**
| Module | Topic | Solution |
|--------|-------|---------|
| 07 — Lakeflow Pipelines | Declarative Bronze→Silver→Gold | Replace manual `dbutils.notebook.run()` with `AUTO CDC` and streaming tables |
| 08 — Orchestration | Schedule + monitor the pipeline | Lakeflow Jobs with DAG, retries, alerts |

The same 6 notebooks you just saw will be deployed as a **Lakeflow Job** with automatic dependency management.

← [05 — Incremental Ingestion](05_incremental_ingestion.ipynb) | **[README](../../../README.md)** | [07 — Lakeflow Pipelines](../../day3/demo/07_lakeflow_pipelines.ipynb) →